# CineScore Phase 5.7: Full Production Suite (Scikit-Learn 1.8 Sync)
This notebook trains the complete Production Suite (Revenue, Acclaim, Showdown) and exports the expanded Reference Database for the Phase 4.7 Dashboard.

In [ ]:
import os
import pandas as pd
import numpy as np
import joblib
import json

from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.neighbors import NearestNeighbors
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.linear_model import ElasticNetCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures, QuantileTransformer

# 1. Environment & Pathing
BASE_PATH = '../Data/Processed_Dataset'
CAST_PATH = '../Data/Raw_Dataset/cast.csv'
CREW_PATH = '../Data/Raw_Dataset/crew.csv'
NLP_PATH = os.path.join(BASE_PATH, 'v8_FINAL_CINESCORE_NLP.csv')
ECON_PATH = os.path.join(BASE_PATH, 'v2_economically_normalized_df.csv')
OUTPUT_DIR = BASE_PATH
print(f'✅ Environment Setup: {OUTPUT_DIR}')

In [ ]:
# 2. Resilient Data Fusion
df_nlp = pd.read_csv(NLP_PATH)
df_econ = pd.read_csv(ECON_PATH)
df_nlp.columns = df_nlp.columns.str.strip()
df_econ.columns = df_econ.columns.str.strip()

redundant_cols = ['title', 'original_title', 'poster_path', 'overview', 'tagline', 'genres', 'four_quadrant_appeal', 'high_concept_marketability']
df_econ_clean = df_econ.drop(columns=[c for c in redundant_cols if c in df_econ.columns])

df = pd.merge(df_nlp[['id', 'title', 'poster_path', 'four_quadrant_appeal', 'high_concept_marketability']], 
              df_econ_clean, on='id', how='inner')

df['budget_per_minute'] = df['log_inflated_budget'] / (df['runtime'] + 1)
df['total_talent_gravity'] = df['actor_1_hpi'] + df['actor_2_hpi'] + df['director_hpi']
df['appeal_budget_interaction'] = df['four_quadrant_appeal'] * df['log_inflated_budget']
df['marketability_budget_interaction'] = df['high_concept_marketability'] * df['log_inflated_budget']

trainable_df = df[df['revenue'] > 1000].copy()
print(f'🔥 Fusion Complete: {trainable_df.shape[0]} Movies synchronized.')

In [ ]:
# 3. Engine A & B: Revenue & Acclaim
poly_cols = ['log_inflated_budget', 'runtime', 'total_talent_gravity']
numeric_features = [
    'log_inflated_budget', 'runtime', 'release_year', 'release_month',
    'budget_per_minute', 'total_talent_gravity',
    'actor_1_hpi', 'actor_2_hpi', 'actor_3_hpi', 'director_hpi', 'producer_hpi', 'writer_hpi',
    'cast_synergy_mult', 'crew_synergy_mult', 'lead_duo_synergy_mult',
    'corenswet_imputation', 'four_quadrant_appeal', 'high_concept_marketability',
    'appeal_budget_interaction', 'marketability_budget_interaction',
    'is_epic_window', 'is_franchise'
]
categorical_features = ['original_language', 'primary_genre', 'primary_studio', 'release_season']

preprocessor = ColumnTransformer([
    ('poly', Pipeline([('pg', PolynomialFeatures(degree=2, include_bias=False)), ('sc', StandardScaler())]), poly_cols),
    ('num', StandardScaler(), [f for f in numeric_features if f not in poly_cols]),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

stacking_regr = StackingRegressor(estimators=[
    ('rf', RandomForestRegressor(n_estimators=750, max_depth=14, random_state=42, n_jobs=-1)),
    ('xgb', XGBRegressor(n_estimators=1000, learning_rate=0.01, max_depth=6, random_state=42, n_jobs=-1)),
    ('lgbm', LGBMRegressor(n_estimators=1000, learning_rate=0.01, max_depth=6, random_state=42, verbose=-1))
], final_estimator=ElasticNetCV(cv=8), cv=5)

# Revenue Model
revenue_pipeline = Pipeline([('pre', preprocessor), ('reg', TransformedTargetRegressor(regressor=stacking_regr, transformer=QuantileTransformer(output_distribution='normal', random_state=42)))])
revenue_pipeline.fit(trainable_df[numeric_features + categorical_features], trainable_df['log_inflated_revenue'])
joblib.dump(revenue_pipeline, os.path.join(OUTPUT_DIR, "v8_cinescore_oracle_PRODUCTION.pkl"))

# Acclaim Model
engine_b = Pipeline([('pre', preprocessor), ('reg', RandomForestRegressor(n_estimators=300, max_depth=10, random_state=42, n_jobs=-1))])
engine_b.fit(trainable_df[numeric_features + categorical_features], trainable_df['vote_average'])
joblib.dump(engine_b, os.path.join(OUTPUT_DIR, "v8_engine_b_acclaim.pkl"))

print("✅ Core Engines Synchronized.")

In [ ]:
# 4. Block 4: EXPANDED Reference Vault & Ledger
print("📦 Building Phase 4.7 Reference Vault...")

df_cast = pd.read_csv(CAST_PATH)[['person_id', 'name']].drop_duplicates()
df_crew = pd.read_csv(CREW_PATH)[['person_id', 'name']].drop_duplicates()

ref_data = trainable_df.copy()
ref_data = ref_data.merge(df_cast.rename(columns={'person_id': 'actor_1_id', 'name': 'actor_1_name'}), on='actor_1_id', how='left')
ref_data = ref_data.merge(df_crew.rename(columns={'person_id': 'director_id', 'name': 'director_name'}), on='director_id', how='left')

ref_cols = [
    'id', 'title', 'poster_path', 'log_inflated_budget', 'revenue', 'vote_average',
    'primary_genre', 'primary_studio', 'release_year', 'actor_1_name', 'director_name'
]
ref_data[ref_cols].to_csv(os.path.join(OUTPUT_DIR, 'v8_reference_database.csv'), index=False)

# Ledger Generation
actors = pd.concat([
    trainable_df[['actor_1_id', 'actor_1_hpi']].rename(columns={'actor_1_id': 'person_id', 'actor_1_hpi': 'hpi'}),
    trainable_df[['actor_2_id', 'actor_2_hpi']].rename(columns={'actor_2_id': 'person_id', 'actor_2_hpi': 'hpi'}),
    trainable_df[['actor_3_id', 'actor_3_hpi']].rename(columns={'actor_3_id': 'person_id', 'actor_3_hpi': 'hpi'})
]).dropna()

directors = trainable_df[['director_id', 'director_hpi']].rename(columns={'director_id': 'person_id', 'director_hpi': 'hpi'})

talent_dict = pd.concat([
    actors.merge(df_cast, on='person_id'), 
    directors.merge(df_crew, on='person_id')
]).groupby('name')['hpi'].mean().to_dict()

with open(os.path.join(OUTPUT_DIR, 'talent_ledger.json'), 'w') as f:
    json.dump(talent_dict, f)

print(f"✅ Final Production Suite Secured: {OUTPUT_DIR}")